<a href="https://colab.research.google.com/github/Hoanghieu003/detection-and-classification-plant-diseases/blob/main/Custom-Pipeline-Detection-Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cài đặt thư viện Ultralytics (nếu chưa có)
!pip install ultralytics -q

import cv2
import numpy as np
from ultralytics import YOLO
from google.colab.patches import cv2_imshow # Hàm hiển thị ảnh an toàn riêng cho Colab
import urllib.request
import os

print("✅ Đã tải xong thư viện!")

In [ ]:
# Hàm phụ để tải ảnh từ link web
def download_image(url, filename='test_image.jpg'):
    try:
        urllib.request.urlretrieve(url, filename)
        return filename
    except Exception as e:
        print(f"Lỗi tải ảnh: {e}")
        return None

# MỘT SỐ LINK ẢNH TEST (Bạn có thể tự thay link URL ảnh khác vào đây)
# Ví dụ: Link một chiếc lá cà chua bị đốm vàng (Early Blight)
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Tomato_leaf_blight.jpg/800px-Tomato_leaf_blight.jpg"

print("Đang tải ảnh test...")
test_img_path = download_image(image_url)

if test_img_path:
    print("Bắt đầu chạy AI Pipeline...")
    run_disease_pipeline(test_img_path)

In [ ]:
# 1. TẢI 2 MÔ HÌNH LÊN BỘ NHỚ RAM CỦA COLAB
# (Nhớ thay đổi tên file nếu bạn đặt tên khác)
print("⏳ Đang nạp mô hình Giai đoạn 1 (Tìm lá)...")
model_stage1 = YOLO('stage1_leaf_detect.pt')

print("⏳ Đang nạp mô hình Giai đoạn 2 (Chẩn đoán bệnh)...")
model_stage2 = YOLO('stage2_disease_class.pt')

print("✅ Hai mô hình đã sẵn sàng!")
print("-" * 50)

# 2. XÂY DỰNG HÀM XỬ LÝ CHÍNH
def run_disease_pipeline(image_path):
    # Đọc ảnh
    img = cv2.imread(image_path)
    if img is None:
        print("❌ Lỗi: Không thể đọc ảnh. Vui lòng kiểm tra lại đường dẫn!")
        return

    # --- GIAI ĐOẠN 1: TÌM LÁ ---
    # conf=0.4: Bỏ qua các vật thể rác, chỉ lấy lá khi AI tự tin > 40%
    results1 = model_stage1(img, conf=0.4, verbose=False)

    leaf_count = 0

    # Duyệt qua các kết quả tìm được
    for r in results1:
        boxes = r.boxes
        for box in boxes:
            leaf_count += 1
            # Tọa độ khung chữ nhật
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Tên giống cây (Ví dụ: Tomato, Apple...)
            plant_id = int(box.cls[0])
            plant_name = model_stage1.names[plant_id]

            # --- CẮT CHIẾC LÁ ---
            leaf_crop = img[y1:y2, x1:x2]

            # Bỏ qua nếu ảnh cắt ra bị lỗi hoặc quá bé (dưới 20 pixel)
            if leaf_crop.shape[0] < 20 or leaf_crop.shape[1] < 20:
                continue

            # --- GIAI ĐOẠN 2: CHẨN ĐOÁN BỆNH ---
            results2 = model_stage2(leaf_crop, verbose=False)
            probs = results2[0].probs

            # Khởi tạo Bộ lọc thông minh (Smart Router)
            best_disease = "Unknown_Disease"
            highest_prob = 0.0

            for idx, prob in enumerate(probs.data):
                disease_name = model_stage2.names[idx]

                # CHỈ DUYỆT CÁC BỆNH CỦA ĐÚNG LOẠI CÂY ĐÓ
                if disease_name.startswith(plant_name):
                    if float(prob) > highest_prob:
                        highest_prob = float(prob)
                        best_disease = disease_name

            # --- VẼ KẾT QUẢ LÊN ẢNH ---
            # Xác định màu sắc (Khỏe = Xanh lá, Bệnh = Đỏ, Không chắc chắn = Cam)
            if highest_prob < 0.4:
                final_label = f"{plant_name} (Uncertain: {highest_prob:.2f})"
                color = (0, 165, 255) # Cam (BBR)
            else:
                final_label = f"{best_disease} ({highest_prob:.2f})"
                if "Healthy" in best_disease:
                    color = (0, 255, 0) # Xanh lá
                else:
                    color = (0, 0, 255) # Đỏ

            # Vẽ khung
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)

            # Vẽ nền cho chữ dễ đọc
            (w, h), _ = cv2.getTextSize(final_label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
            cv2.rectangle(img, (x1, y1 - 30), (x1 + w, y1), color, -1)

            # Ghi chữ (Màu trắng)
            cv2.putText(img, final_label, (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    if leaf_count == 0:
         print("⚠️ Không tìm thấy chiếc lá nào trong bức ảnh này.")
    else:
         print(f"✅ Đã xử lý xong! Tìm thấy {leaf_count} chiếc lá.")

    # Hiển thị ảnh ngay trên Colab
    cv2_imshow(img)

    # Tùy chọn: Lưu ảnh ra file
    # cv2.imwrite('result.jpg', img)